In [ ]:
import matplotlib.pyplot as plt

#解决中文显示问题
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import pickle
import networkx as nx
import matplotlib.pyplot as plt

# 读取存储的结构计数文件
structure_counts_file_path = 'D:/Biyesheji1/structure_counts.pkl'
with open(structure_counts_file_path, 'rb') as f:
    structure_counts = pickle.load(f)


max_key = min(structure_counts, key=structure_counts.get)
max_key

In [ ]:
len(structure_counts.keys())

In [ ]:
# 读取存储的结构计数文件
Sum_graphs_file_path = 'D:/Biyesheji1/Sum_graphs.pkl'
with open(Sum_graphs_file_path, 'rb') as f:
    Sum_graphs = pickle.load(f)

In [ ]:
list(structure_counts.keys())

In [ ]:
# 绘制图结构
pos = nx.spring_layout(max_key)  # 选择一个布局算法，这里选择 Spring layout
nx.draw(max_key, pos, with_labels=True, node_size=700, node_color="skyblue", font_size=12, font_weight="bold")
plt.title('数量最多的结构')
plt.show()

In [ ]:
# 对字典的值进行排序
sorted_values = sorted(structure_counts.values())

# 获取第二大的值
second_largest_value = sorted_values[1]

second_largest_value2 = sorted_values[0]

# 找到第二大值对应的键
second_largest_key = [key for key, value in structure_counts.items() if value == second_largest_value]

# 找到第二大值对应的键
second_largest_key2 = [key for key, value in structure_counts.items() if value == second_largest_value2]

In [ ]:
second_largest_key[0].edges()

In [ ]:
# 绘制图结构
pos = nx.spring_layout(second_largest_key[8])  # 选择一个布局算法，这里选择 Spring layout
nx.draw(second_largest_key[0], pos, with_labels=True, node_size=700, node_color="skyblue", font_size=12, font_weight="bold")
plt.title('数量次多的结构')
plt.show()

In [ ]:
nx.is_isomorphic(second_largest_key[7], second_largest_key[6])

In [ ]:
import pandas as pd

temp_data=pd.DataFrame(list(structure_counts.values()))
temp_data

In [ ]:
temp_data.sort_values(by=0,ascending=False)

In [ ]:
list(temp_data[0])

In [ ]:
list(structure_counts.keys())

In [ ]:
G=list(structure_counts.keys())[0]

In [ ]:
# 输出节点的 "category" 属性
for node in G.nodes:
    if 'category' in G.nodes[node]:
        print("Node {}: category = {}".format(node, G.nodes[node]['category']))
    else:
        print("Node {}: does not have 'category' attribute".format(node))

In [ ]:
list(Sum_graphs.values())

In [ ]:
import networkx as nx

# 给定的汉字与数字的映射关系
chinese_to_number = {"中坚支持者": 0, "中介成员": 1, "最近参与成员": 2,'创新突破者':3,'领导者':4}

# 将节点属性按照给定的字典转换为数字
for node, category in nx.get_node_attributes(list(Sum_graphs.values())[0], "category").items():
    list(Sum_graphs.values())[0].nodes[node]["category"] = chinese_to_number[category]

# 将节点名称转换为数字
G_int = nx.convert_node_labels_to_integers(list(Sum_graphs.values())[0], first_label=0, ordering='default', label_attribute=None)

# 构建节点名称与数字对应的字典
mapping = {node: i for i, node in enumerate(list(Sum_graphs.values())[0].nodes)}

# 输出节点的映射关系
print("节点名称与数字对应的字典:")
print(mapping)
mapping

# 输出转换后的图的节点
print("\n转换后的图的节点:")
print(G_int.nodes())

In [ ]:
import pandas as pd

# 将字典转换为 DataFrame
df = pd.DataFrame(list(mapping.items()), columns=['Node', 'Number'])
df

In [ ]:
import networkx as nx

def graph_to_data_file(graph, file_path):
    with open(file_path, 'w') as file:
        print("t # 0\n")
        file.write("t # 0\n")
        
        # 写入节点信息
        for node, data in graph.nodes(data=True):
            category = data.get('category', 'None')  # 获取节点的 'category' 属性，如果不存在则默认为 'None'
            file.write(f"v {node} {category}\n")
            print(f"v {node} {category}\n")
            
        
        # 写入边信息
        for edge in graph.edges(data=True):
            source, target, data = edge
            weight = data.get('weight', 1)  # 获取边的 'weight' 属性，如果不存在则默认为 1
            file.write(f"e {source} {target} {int(weight)}\n")
        
        file.write("t # -1\n")
        print("t # -1\n")

graph_to_data_file(G_int, "output_data.txt")

In [ ]:
import pandas as pd
import pickle
import networkx as nx
import matplotlib.pyplot as plt


def Node_data_bulid(df_sum,name):
    # 定义有向图
    DG = nx.DiGraph()

    for index, row in df_sum.iterrows():
        row = df_sum.loc[index]
        DG.add_edges_from([(row['Source'], row['Target'], {'weight': row['Weight']})])

    nodes_data=pd.DataFrame(DG.nodes())
    nodes_data.columns=['username']

    Node_data=pd.merge(nodes_data,(df[df['File_Name']==name])[['username','result']],on='username',how='left')

    Node_data = Node_data.dropna()
    Node_data.reset_index(drop=True, inplace=True)

    return Node_data


#解决中文显示问题
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus'] = False

def net_build(Node_data,df_sum):
    # 创建一个有向图
    new_G = nx.DiGraph()

    # 添加节点
    for index, row in Node_data.iterrows():
        new_G.add_node(row['username'], category=row['result'])

    for index,row in df_sum.iterrows():
        row=df_sum.loc[index]
        new_G.add_edges_from([(row['Source'],row['Target'], {'weight':row['Weight']})])

    return new_G

def NET_Build(i):
    temp_data=pd.read_csv(f'D:/Biyesheji1/edge_data1/{i}.csv')

    # 按起始点和终点分组，并对权重进行累加
    df_sum = temp_data.groupby(['Source', 'Target'], as_index=False).agg({'Weight': 'sum', 'Time': 'max'})
    # 输出处理后的DataFrame

    Node_data=Node_data_bulid(df_sum, i)

    columns_to_check = ['Source', 'Target']  # 要检查的列

    # 创建一个布尔索引，选择列中的值在允许的值列表中的行
    mask = df_sum[columns_to_check].apply(lambda x: x.isin(list(Node_data['username']))).all(axis=1)

    # 根据布尔索引选择满足条件的行
    df_sum = df_sum[mask]

    new_G=net_build(Node_data, df_sum)
    
    return new_G

In [ ]:
df = pd.read_csv('D:/jupyter_notebook/毕业设计/动态聚类结果.csv')

In [ ]:
from igraph import Graph

def net_to_graph(nx_graph):
    # 将NetworkX有向图转换为igraph图
    ig_graph = Graph(directed=True)
    node_map = {node: idx for idx, node in enumerate(set(nx_graph.nodes()))}  # 创建节点映射
    ig_graph.add_vertices(len(node_map))  # 添加节点
    ig_graph.add_edges([(node_map[src], node_map[dst]) for src, dst in nx_graph.edges()])  # 添加边
    
    return graph_igraph

In [ ]:
# 创建一个有向图
nx_graph = nx.DiGraph()
nx_graph.add_nodes_from([1, 2, 3])
nx_graph.add_edges_from([(1, 2), (2, 3)])

In [ ]:
net_to_graph(nx_graph)

In [ ]:

import networkx as nx
import igraph
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
#import public_motif as motif

def motif_mining(graph_igraph,motif_igraph):
    mappings = graph_igraph.get_subisomorphisms_lad(motif_igraph, induced=True) 
    mappings_set = set(tuple(sorted(i)) for i in mappings)
    """for i in mappings_set:
        print(i)"""
        
    return mappings_set


In [ ]:
list(df['File_Name'].unique())

In [ ]:
nx_graph=NET_Build(list(df['File_Name'].unique())[0])

In [ ]:
# 输出节点属性
for node, attrs in nx_graph.nodes(data=True):
    print("Node:", node, "Attributes:", attrs)

In [ ]:

for i in list(df['File_Name'].unique()):
    nx_graph=NET_Build(i)
    print(nx_graph)
    graph_igraph=net_to_graph(nx_graph)
    for j in list(structure_counts.keys()):
        print("======================================")
        motif_nx=j
        motif_igraph=net_to_graph(motif_nx)
        mappings_set=motif_mining(graph_igraph,motif_igraph)
        print(len(mappings_set))

In [ ]:
NET_Graph=[]
for i in list(df['File_Name'].unique()):
    print(i)
    nx_graph=NET_Build(i)
    NET_Graph.append(nx_graph)

In [ ]:
def net_to_graph(nx_graph):
    # 将NetworkX有向图转换为igraph图
    ig_graph = Graph(directed=True)
    
    # 创建节点映射
    node_map = {node: idx for idx, node in enumerate(set(nx_graph.nodes()))}
    
    # 添加节点并记录节点类型
    node_types = {}
    for node, idx in node_map.items():
        node_type = nx_graph.nodes[node].get('category', None)  # 获取节点的 'category' 属性，如果不存在则为 None
        ig_graph.add_vertex(name=str(idx), category=node_type)  # 将节点属性添加到 igraph 图的节点中
        node_types[str(idx)] = node_type
    
    # 添加边
    for src, dst in nx_graph.edges():
        src_idx = node_map[src]
        dst_idx = node_map[dst]
        ig_graph.add_edge(str(src_idx), str(dst_idx))

    return ig_graph, node_types

def motif_mining(graph_igraph, motif_igraph, node_types):
    mappings = graph_igraph.get_subisomorphisms_lad(motif_igraph, induced=True)
    mappings_set = set()
    
    # 仅保留节点类型匹配的映射
    for mapping in mappings:
        if all(node_types[node] == motif_igraph.vs[mapping[idx]]['category'] for idx, node in enumerate(mapping)):
            mappings_set.add(tuple(sorted(mapping)))
    
    return mappings_set

# 假设存在一个 df DataFrame，其中包含网络文件的 File_Name 和节点类型的信息
for file_name in df['File_Name'].unique():
    nx_graph = NET_Build(file_name)
    graph_igraph, node_types = net_to_graph(nx_graph)
    
    for motif_nx, count in structure_counts.items():
        motif_igraph, _ = net_to_graph(motif_nx)
        mappings_set = motif_mining(graph_igraph, motif_igraph, node_types)
        
        print("Number of occurrences of motif in graph:", len(mappings_set))